In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import subprocess, os, shutil

USB_PATH = '/kaggle/working/USB'
CG_PATH  = '/kaggle/working/CGMatch'

os.system(f"rm -rf {USB_PATH} {CG_PATH}")
os.system(f"git clone https://github.com/microsoft/Semi-supervised-learning {USB_PATH}")
os.system(f"git clone https://github.com/Alam8010/CGMatch.git {CG_PATH}")

shutil.copytree(f'{CG_PATH}/semilearn', f'{USB_PATH}/semilearn', dirs_exist_ok=True)
shutil.copytree(f'{CG_PATH}/config',    f'{USB_PATH}/config',    dirs_exist_ok=True)
shutil.copy(f'{CG_PATH}/train.py', f'{USB_PATH}/train.py')

os.system("pip install timm==0.6.12 progress ruamel.yaml tensorboard tqdm scikit-learn -q")

r = subprocess.run(
    ['python', '-c', 'from semilearn.algorithms import get_algorithm; print("Import OK ✅")'],
    cwd=USB_PATH, capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())

Cloning into '/kaggle/working/USB'...
Cloning into '/kaggle/working/CGMatch'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 13.8 MB/s eta 0:00:00
Import OK ✅


In [2]:
with open('/kaggle/working/USB/config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if   'num_train_iter' in line: new_lines.append('num_train_iter: 16384\n')
    elif 'warm_up_iter'   in line: new_lines.append('warm_up_iter: 32\n')
    elif 'num_eval_iter'  in line: new_lines.append('num_eval_iter: 512\n')
    elif line.strip().startswith('gpu:'): new_lines.append('gpu: 0\n')
    else: new_lines.append(line)

with open('/kaggle/working/USB/config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml', 'w') as f:
    f.writelines(new_lines)

print("YAML patched ✅")

YAML patched ✅


In [3]:
!cd /kaggle/working/USB && python train.py \
    --c config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml \
    --save_dir /kaggle/working/results

/kaggle/working/USB/train.py:286: UserWarning: You have chosen to seed training. This will turn on the CUDNN deterministic setting, which can slow down your training considerably! You may see unexpected behavior when restarting from checkpoints.
  warnings.warn(
/kaggle/working/USB/train.py:297: UserWarning: You have chosen a specific GPU. This will completely disable data parallelism.
  warnings.warn(
[2026-10-19 16:32:58,997 INFO] Use GPU: 0 for training
/kaggle/working/USB/semilearn/core/algorithmbase.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.loss_scaler = GradScaler()
100%|████████████████████████████████████████| 170M/170M [52:07<00:00, 54.5kB/s]
lb count: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
ulb count: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
[2026-10-19 17:25:11,184 INFO] unlabeled data number: 50000, labeled data number 40
[2026-10-19 17:25:11,185 INFO] Create train 

In [4]:
import subprocess, os, shutil

USB_PATH = '/kaggle/working/USB'
CG_PATH  = '/kaggle/working/CGMatch'

os.system(f"rm -rf {USB_PATH} {CG_PATH}")
os.system(f"git clone https://github.com/microsoft/Semi-supervised-learning {USB_PATH}")
os.system(f"git clone https://github.com/Alam8010/CGMatch.git {CG_PATH}")

shutil.copytree(f'{CG_PATH}/semilearn', f'{USB_PATH}/semilearn', dirs_exist_ok=True)
shutil.copytree(f'{CG_PATH}/config',    f'{USB_PATH}/config',    dirs_exist_ok=True)
shutil.copy(f'{CG_PATH}/train.py', f'{USB_PATH}/train.py')

os.system("pip install timm==0.6.12 progress ruamel.yaml tensorboard tqdm scikit-learn -q")

r = subprocess.run(
    ['python', '-c', 'from semilearn.algorithms import get_algorithm; print("Import OK ✅")'],
    cwd=USB_PATH, capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())

Cloning into '/kaggle/working/USB'...
Cloning into '/kaggle/working/CGMatch'...


Import OK ✅


In [5]:
with open('/kaggle/working/USB/semilearn/algorithms/cgmatch/cgmatch.py', 'r') as f:
    c = f.read()

# Add random import if not already there
if 'import random' not in c:
    c = 'import random\n' + c

# Find and replace the count_gap based easy/ambiguous/hard assignment with random
import re

# Pattern: replaces the if/elif block that checks count_gap values
# with a random assignment to one of the three buckets
old_pattern = r'(if.*count_gap.*:.*\n.*easy.*\n.*elif.*count_gap.*:.*\n.*ambiguous.*\n.*else.*:.*\n.*hard)'
new_pattern = '''bucket = random.choice(['easy', 'ambiguous', 'hard'])
            if bucket == 'easy':
                pass  # handled below as easy
            elif bucket == 'ambiguous':
                pass  # handled below as ambiguous
            else:
                pass  # handled below as hard'''

# Simpler approach - just find the FDS function and show what we're working with
fds_idx = c.find('def FDS(')
if fds_idx == -1:
    print("FDS not found — printing count_gap references instead:")
    for i, line in enumerate(c.split('\n')):
        if 'count_gap' in line.lower() or 'easy' in line.lower() and 'ambiguous' in line.lower():
            print(f"Line {i}: {line}")
else:
    print("FDS function found. First 800 chars:")
    print(c[fds_idx:fds_idx+800])

FDS not found — printing count_gap references instead:
Line 53:         self.count_gap = {}
Line 54:         self.count_gap_acc = {}
Line 86:     def compute_count_gap(self, ulb_idxs=None):
Line 94:         count_gaps = []
Line 106:             count_gaps.append(difference)
Line 108:         return count_gaps, max_values
Line 127:         count_gaps, max_values = self.compute_count_gap(ulb_idxs)
Line 132:             self.count_gap[guid] = count_gaps[i]
Line 135:                 self.count_gap_acc[guid] = 1 if self.guid_gold[guid]['gold'] == max_values[i] else 0
Line 137:                 self.count_gap_acc[guid] = 0
Line 139:         return count_gaps
Line 150:                   'count_gap',
Line 151:                   'count_gap_acc',]
Line 157:                             self.count_gap[guid],
Line 158:                             self.count_gap_acc[guid],
Line 237:             count_gaps = self.compute_train_dy_metrics(idx_ulb)
Line 240:             easy_mask, ambiguous_mask = self.

In [6]:
with open('/kaggle/working/USB/semilearn/algorithms/cgmatch/cgmatch.py', 'r') as f:
    c = f.read()

old_line = 'easy_mask, ambiguous_mask = self.call_hook("masking", "MaskingHook", logits_x_ulb=probs_x_ulb_w, idx_ulb=idx_ulb, softmax_x_ulb=False)'

new_line = '''rand_vals = torch.rand(probs_x_ulb_w.shape[0]).to(probs_x_ulb_w.device)
            easy_mask = rand_vals < 1/3
            ambiguous_mask = (rand_vals >= 1/3) & (rand_vals < 2/3)
            # hard samples: rand_vals >= 2/3 (excluded from both losses)'''

if old_line in c:
    c = c.replace(old_line, new_line)
    with open('/kaggle/working/USB/semilearn/algorithms/cgmatch/cgmatch.py', 'w') as f:
        f.write(c)
    print("Random sorting patch applied ✅")
else:
    print("ERROR: line not found — paste this output here")

Random sorting patch applied ✅


In [7]:
with open('/kaggle/working/USB/config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml', 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if   'num_train_iter' in line: new_lines.append('num_train_iter: 16384\n')
    elif 'warm_up_iter'   in line: new_lines.append('warm_up_iter: 32\n')
    elif 'num_eval_iter'  in line: new_lines.append('num_eval_iter: 512\n')
    elif line.strip().startswith('gpu:'): new_lines.append('gpu: 0\n')
    else: new_lines.append(line)

with open('/kaggle/working/USB/config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml', 'w') as f:
    f.writelines(new_lines)

print("YAML patched ✅")

YAML patched ✅


In [ ]:
!cd /kaggle/working/USB && CUDA_VISIBLE_DEVICES=0 python train.py \
    --c config/classic_cv/cgmatch/cgmatch_cifar10_40_0.yaml \
    --save_dir /kaggle/working/results_random

/kaggle/working/USB/train.py:286: UserWarning: You have chosen to seed training. This will turn on the CUDNN deterministic setting, which can slow down your training considerably! You may see unexpected behavior when restarting from checkpoints.
  warnings.warn(
/kaggle/working/USB/train.py:297: UserWarning: You have chosen a specific GPU. This will completely disable data parallelism.
  warnings.warn(
[2026-10-19 21:12:44,557 INFO] Use GPU: 0 for training
/kaggle/working/USB/semilearn/core/algorithmbase.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.loss_scaler = GradScaler()
100%|████████████████████████████████████████| 170M/170M [43:05<00:00, 65.9kB/s]
lb count: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
ulb count: [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
[2026-10-19 21:55:54,605 INFO] unlabeled data number: 50000, labeled data number 40
[2026-10-19 21:55:54,605 INFO] Create train 